# Inspect Model Files and Data

This notebook loads datasets and model artifacts from `data/` and `models/`, summarizes what is inside each file,
and evaluates **model1** for each dataset using accuracy, precision, recall, and F1. Results are plotted with Plotly.

In [ ]:
from pathlib import Path
import os
import sys
import json
import pickle
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / 'data'
MODELS_DIR = ROOT / 'models'
EXPLAIN_DIR = ROOT / 'explainers'

print('ROOT:', ROOT)
print('CWD:', Path.cwd())
print('DATA_DIR exists:', DATA_DIR.exists())
print('MODELS_DIR exists:', MODELS_DIR.exists())
print('EXPLAIN_DIR exists:', EXPLAIN_DIR.exists())

In [ ]:
def _try_import_torch():
    try:
        import torch  # noqa: F401
        return True
    except Exception as exc:
        print('Torch not available:', exc)
        return False

TORCH_AVAILABLE = _try_import_torch()

def load_pt(path: Path):
    path = Path(path)
    if TORCH_AVAILABLE:
        import torch
        return torch.load(path, map_location='cpu')
    # Fallback: pickle
    with open(path, 'rb') as f:
        return pickle.load(f)

def load_joblib(path: Path):
    import joblib
    return joblib.load(path)

def load_any(path: Path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == '.npy':
        return np.load(path, allow_pickle=True)
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix == '.pt':
        return load_pt(path)
    if suffix == '.joblib':
        return load_joblib(path)
    # Fallback: pickle
    with open(path, 'rb') as f:
        return pickle.load(f)

def summarize_object(obj, max_items=5):
    info = {'type': type(obj).__name__}
    if isinstance(obj, np.ndarray):
        info.update({'shape': obj.shape, 'dtype': str(obj.dtype)})
    elif isinstance(obj, (list, tuple)):
        info.update({'len': len(obj)})
    elif isinstance(obj, dict):
        keys = list(obj.keys())
        info.update({'len': len(obj), 'keys': keys[:max_items]})
    else:
        # Try torch tensor
        if TORCH_AVAILABLE:
            import torch
            if isinstance(obj, torch.Tensor):
                info.update({'shape': tuple(obj.shape), 'dtype': str(obj.dtype)})
    return info

def summarize_model(obj, max_items=5):
    info = {'type': type(obj).__name__}
    if TORCH_AVAILABLE:
        import torch
        if isinstance(obj, torch.nn.Module):
            n_params = sum(p.numel() for p in obj.parameters())
            info.update({'n_params': int(n_params)})
            return info
    if isinstance(obj, dict):
        # Likely a state_dict or metadata bundle
        keys = list(obj.keys())
        info.update({'len': len(obj), 'keys': keys[:max_items]})
        return info
    # Fallback
    return info

In [ ]:
datasets = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
datasets

In [ ]:
def load_dataset(name: str, split: str = 'test'):
    ddir = DATA_DIR / name
    x_path = ddir / f'X{split}.pt'
    y_path = ddir / f'y{split}.pt'
    if not x_path.exists() or not y_path.exists():
        raise FileNotFoundError(f'Missing {x_path} or {y_path}')
    X = load_any(x_path)
    y = load_any(y_path)
    return X, y

def summarize_dataset(name: str, split: str = 'test'):
    X, y = load_dataset(name, split=split)
    return {
        'dataset': name,
        'split': split,
        'X': summarize_object(X),
        'y': summarize_object(y),
    }

summaries = [summarize_dataset(d, split='test') for d in datasets]
pd.DataFrame(summaries)

In [ ]:
def list_models():
    return sorted(MODELS_DIR.glob('*.pt'))

model_paths = list_models()
model_paths[:5]

In [ ]:
def parse_model_name(path: Path):
    # Format: <dataset>_model<idx>.pt
    stem = path.stem
    if '_model' in stem:
        dataset, suffix = stem.split('_model', 1)
        return dataset, suffix
    return stem, ''

rows = []
for path in model_paths:
    dataset, model_id = parse_model_name(path)
    obj = load_any(path)
    rows.append({
        'dataset': dataset,
        'model_id': model_id,
        'file': path.name,
        'summary': summarize_model(obj),
    })
pd.DataFrame(rows)

In [ ]:
# Optional: peek into a dataset's CSV if present
def load_csv_preview(name: str, n=5):
    csv_path = DATA_DIR / name / f'{name}_csv.csv'
    if not csv_path.exists():
        return None
    df = pd.read_csv(csv_path)
    return df.head(n)

load_csv_preview(datasets[0])

## Class imbalance

In [ ]:
import plotly.express as px

def _to_numpy(x):
    if TORCH_AVAILABLE:
        import torch
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
    return np.asarray(x)

def _prepare_y(y):
    y = _to_numpy(y)
    if y.ndim == 2 and y.shape[1] > 1:
        return y.argmax(axis=1)
    return y.reshape(-1)

def class_distribution_for_dataset(dataset: str, split: str = 'test'):
    _, y = load_dataset(dataset, split=split)
    y_true = _prepare_y(y)
    classes, counts = np.unique(y_true, return_counts=True)
    total = counts.sum()
    rows = []
    for c, n in zip(classes, counts):
        rows.append({
            'dataset': dataset,
            'split': split,
            'class': int(c),
            'count': int(n),
            'percent': float(n / total) if total > 0 else 0.0,
        })
    return rows

dist_rows = []
for d in datasets:
    dist_rows.extend(class_distribution_for_dataset(d, split='test'))

dist_df = pd.DataFrame(dist_rows)
dist_df

In [ ]:
if len(dist_df) > 0:
    fig = px.bar(
        dist_df,
        x='dataset',
        y='count',
        color='class',
        barmode='stack',
        title='Class Counts by Dataset (Test Split)',
    )
    fig.show()

    fig2 = px.bar(
        dist_df,
        x='dataset',
        y='percent',
        color='class',
        barmode='stack',
        title='Class Percent by Dataset (Test Split)',
        range_y=[0, 1],
    )
    fig2.show()
else:
    print('No class distributions to plot.')

## Evaluate model1 per dataset

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def _predict_labels(model, X):
    if not TORCH_AVAILABLE:
        raise RuntimeError('Torch is required to run model inference.')
    if isinstance(X, np.ndarray):
        X_t = torch.tensor(X, dtype=torch.float32)
    elif isinstance(X, torch.Tensor):
        X_t = X.float()
    else:
        X_t = torch.tensor(np.asarray(X), dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        out = model(X_t)
    out = _to_numpy(out)

    # Binary vs multiclass handling
    if out.ndim == 1:
        probs = 1 / (1 + np.exp(-out))  # sigmoid
        return (probs >= 0.5).astype(int)
    if out.ndim == 2 and out.shape[1] == 1:
        probs = 1 / (1 + np.exp(-out.reshape(-1)))
        return (probs >= 0.5).astype(int)
    # Multiclass logits or probabilities
    return out.argmax(axis=1)

def evaluate_model1_for_dataset(dataset: str):
    try:
        X, y = load_dataset(dataset, split='test')
        y_true = _prepare_y(y)

        model = load_model_for_dataset(dataset, 'model1')
        y_pred = _predict_labels(model, _to_numpy(X))
        classes = np.unique(y_true)
        average = 'binary' if classes.size == 2 else 'macro'

        return {
            'dataset': dataset,
            'status': 'ok',
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, average=average, zero_division=0),
            'recall': recall_score(y_true, y_pred, average=average, zero_division=0),
            'f1': f1_score(y_true, y_pred, average=average, zero_division=0),
            'n_samples': int(len(y_true)),
            'n_classes': int(classes.size),
        }
    except Exception as exc:
        return {
            'dataset': dataset,
            'status': 'error',
            'error': str(exc),
        }

results = [evaluate_model1_for_dataset(d) for d in datasets]
results_df = pd.DataFrame(results)
results_df

In [ ]:
ok_df = results_df[results_df['status'] == 'ok'].copy()
ok_df

In [ ]:
if len(ok_df) > 0:
    long_df = ok_df.melt(
        id_vars=['dataset'],
        value_vars=['accuracy', 'precision', 'recall', 'f1'],
        var_name='metric',
        value_name='value',
    )
    fig = px.bar(
        long_df,
        x='dataset',
        y='value',
        color='metric',
        barmode='group',
        title='Model1 Metrics by Dataset',
        range_y=[0, 1],
    )
    fig.show()
else:
    print('No successful evaluations to plot. Check errors above.')